# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# ML-08 - Structured Content Archetype Clustering
# Method choice

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import matplotlib.pyplot as plt

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("""
Method choice:
I use K-Means clustering because the goal of this lane is to discover
recurring performance archetypes across the content inventory.

I scale the numeric features first so variables with larger numeric
ranges do not dominate the distance calculation. I use PCA only for
two-dimensional visualization and interpretation.

This is metric-based clustering, not semantic clustering, because the
dataset does not contain article text.
""")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

features = [c for c in candidate_features if c in df.columns]

print("Features used:")
print(features)

X = df[features].copy()

# Clean numeric values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nRows used:", len(X))
print("Feature count:", len(features))

print("""
Validation design:
Because this is unsupervised clustering, there is no target label to
split into train and test. Instead, I check whether the discovered
clusters remain similar across different random seeds.
""")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# Train clustering model

K = 4

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20
)

df["cluster"] = kmeans.fit_predict(X_scaled)

silhouette = silhouette_score(X_scaled, df["cluster"])

print("Number of clusters:", K)
print("Silhouette score:", round(silhouette, 4))

# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df["pca_1"] = X_pca[:, 0]
df["pca_2"] = X_pca[:, 1]

# Cluster profile
profile_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

profile_cols = [c for c in profile_cols if c in df.columns]

cluster_profile = (
    df.groupby("cluster")[profile_cols]
      .median()
      .round(2)
)

cluster_counts = df["cluster"].value_counts().sort_index()

cluster_profile.insert(0, "pages", cluster_counts)

print("\nCluster profile:")
display(cluster_profile)

# Compare with Week-4 baseline if available
if "baseline_refresh_score" in df.columns:
    print("\nBaseline comparison:")
    print(
        df.groupby("cluster")["baseline_refresh_score"]
          .agg(["count", "mean", "median", "min", "max"])
          .round(3)
    )
else:
    print("""
Week-4 baseline score is not present in this input file.
The clusters are therefore interpreted from their own performance
profiles rather than treated as predictions of the baseline.
""")

# PCA plot
plt.figure(figsize=(9, 6))
plt.scatter(
    df["pca_1"],
    df["pca_2"],
    c=df["cluster"],
    alpha=0.5
)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Content Performance Archetype Clusters")
plt.show()

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.